In [1]:
from ultralytics import YOLO
import cv2
import numpy as np

In [2]:
# Load YOLOv8 pose model
model = YOLO('yolov8n-pose.pt')

In [12]:
video_path = 'adeka_upal.avi'  # <-- Replace with your video file
cap = cv2.VideoCapture(video_path)

In [13]:
output_txt = open("yolo_skeleton_coordinates.txt", "w")

In [14]:
frame_id = 0

In [15]:
def is_valid(pt):
    return pt is not None and not np.isnan(pt[0]) and not np.isnan(pt[1])

In [16]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(source=frame, save=False, conf=0.3, verbose=False)
    
    for result in results:
        keypoints = result.keypoints
        if keypoints is not None:
            for person in keypoints.xy:
                coords = person.cpu().numpy()

                try:
                    # Get required keypoints
                    l_hip = coords[11]
                    r_hip = coords[12]
                    l_knee = coords[13]
                    r_knee = coords[14]
                    l_ankle = coords[15]
                    r_ankle = coords[16]
                    l_shoulder = coords[5]
                    r_shoulder = coords[6]

                    # Compute spine top and bottom
                    spine_top = (l_shoulder + r_shoulder) / 2
                    spine_bottom = (l_hip + r_hip) / 2

                    # Save coordinates
                    output_txt.write(f"{frame_id},{spine_top[0]},{spine_top[1]},{spine_bottom[0]},{spine_bottom[1]},"
                                     f"{l_hip[0]},{l_hip[1]},{l_knee[0]},{l_knee[1]},{l_ankle[0]},{l_ankle[1]},"
                                     f"{r_hip[0]},{r_hip[1]},{r_knee[0]},{r_knee[1]},{r_ankle[0]},{r_ankle[1]}\n")

                    # Draw spine
                    if is_valid(spine_top) and is_valid(spine_bottom):
                        cv2.line(frame, spine_top.astype(int), spine_bottom.astype(int), (255, 0, 0), 2)

                    # Left leg
                    if is_valid(l_hip) and is_valid(l_knee):
                        cv2.line(frame, l_hip.astype(int), l_knee.astype(int), (0, 255, 255), 2)
                    if is_valid(l_knee) and is_valid(l_ankle):
                        cv2.line(frame, l_knee.astype(int), l_ankle.astype(int), (255, 255, 0), 2)

                    # Right leg
                    if is_valid(r_hip) and is_valid(r_knee):
                        cv2.line(frame, r_hip.astype(int), r_knee.astype(int), (0, 255, 255), 2)
                    if is_valid(r_knee) and is_valid(r_ankle):
                        cv2.line(frame, r_knee.astype(int), r_ankle.astype(int), (255, 255, 0), 2)

                except Exception as e:
                    print(f"Skipping frame {frame_id} due to error: {e}")

    frame_id += 1

    # Uncomment to preview drawing live
    # cv2.imshow("Skeleton", frame)
    # if cv2.waitKey(1) & 0xFF == ord('q'):
    #     break

cap.release()
output_txt.close()
cv2.destroyAllWindows()